# 🛠️ LoRA Fine-Tuning & Evaluation with Hugging Face, PEFT & W&B

This notebook walks through the end-to-end **Supervised Fine-Tuning (SFT)** workflow using **Hugging Face Transformers**, **PEFT (LoRA)**, and **Weights & Biases (W&B)** lineage tracking:
1. **Prerequisite Check:** Validate CUDA availability, VRAM, and configuration paths.
2. **LoRA Training & W&B Tracking:** Consume the curated SFT dataset artifact (`curated-sft-dataset:latest` when available), attach PEFT adapters, log training/evaluation metrics, and publish the fine-tuned LoRA adapter artifact.
3. **Local Evaluation:** Classification report and confusion matrix.
4. **W&B Evaluation Run:** Dedicated W&B run (`job_type="evaluation"`) with classification metrics and an interactive **W&B Table**.

In [ ]:
# Import required libraries for training, PEFT, tokenization, logging, and metrics
import os
import sys
import torch
import logging
import json
from omegaconf import DictConfig, OmegaConf
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig
from huggingface_hub import snapshot_download
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import wandb

# Configure logging for clear status messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

PROJECT_ROOT = "/data/nemo-fraud-detection"
print("✅ Libraries loaded successfully.")

### 🔍 Step 1: Validate Prerequisites
Checks whether a CUDA-capable GPU is available, inspects free VRAM, and verifies required configuration fields.

In [ ]:
def validate_prerequisites(cfg: DictConfig) -> None:
    logging.info("🔍 Validating training prerequisites...")

    if not torch.cuda.is_available():
        raise RuntimeError("❌ No CUDA-capable GPU found! Training requires a GPU.")
    
    vram_free_gb = torch.cuda.mem_get_info()[0] / (1024**3)
    logging.info(f"✅ GPU detected: {torch.cuda.get_device_name(0)} (Free VRAM: {vram_free_gb:.2f} GB)")

    model_path = cfg.model.get("restore_from_path", None)
    if not model_path:
        raise ValueError("❌ Missing 'cfg.model.restore_from_path' in configuration!")

    logging.info("✅ Prerequisites validated successfully.")

### Step 2: LoRA Fine-Tuning Pipeline & W&B Lineage
Loads the base model, applies PEFT configuration, consumes or resolves the curated SFT splits, logs training/validation metrics to W&B, and registers the trained LoRA adapter artifact.

In [ ]:
def run_training_and_evaluation(cfg: DictConfig) -> bool:
    run = None
    try:
        validate_prerequisites(cfg)

        model_path = cfg.model.restore_from_path
        if model_path.startswith("hf://"):
            model_path = model_path.replace("hf://", "")

        if not os.path.exists(model_path) and not "/" in model_path:
            logging.info(f"📥 Downloading Hugging Face model '{model_path}'...")
            model_path = snapshot_download(repo_id=model_path)

        save_path = cfg.model.get("save_to", "results/fraud_detection_lora")
        abs_save_path = save_path if os.path.isabs(save_path) else os.path.join(PROJECT_ROOT, save_path)

        micro_batch = cfg.model.get("data", {}).get("train_ds", {}).get("micro_batch_size", 1)
        learning_rate = cfg.model.get("optim", {}).get("lr", 2e-4)
        max_steps = cfg.model.get("max_steps", 100)

        run = wandb.init(
            project="nemo-fraud-detection",
            job_type="fine-tuning",
            name="lora-hf-sft-training",
            tags=["stage-3-finetuning", "lora", "peft", "lineage"],
            config={
                "pipeline": "hf-peft-lora",
                "base_model": model_path,
                "lora_r": cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dim", 16),
                "lora_alpha": cfg.model.get("peft", {}).get("qlora_tuning", {}).get("lora_alpha", 32),
                "lora_dropout": cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dropout", 0.05),
                "micro_batch_size": micro_batch,
                "gradient_accumulation_steps": 4,
                "learning_rate": learning_rate,
                "max_steps": max_steps,
                "input_artifact": "curated-sft-dataset:latest",
            }
        )

        logging.info(f"📥 Initializing standard 16-bit LoRA model from: {model_path}...")

        # 1. Load base model in 16-bit precision
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        # 2. Define LoRA (PEFT) configuration
        peft_config = LoraConfig(
            r=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dim", 16),
            lora_alpha=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("lora_alpha", 32),
            target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc_in", "fc_out", "w1", "w2"],
            lora_dropout=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dropout", 0.05),
            bias="none",
            task_type="CAUSAL_LM"
        )

        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        trainable_pct = (trainable_params / total_params) * 100 if total_params > 0 else 0.0
        run.log({
            "train/trainable_params": trainable_params,
            "train/total_params": total_params,
            "train/trainable_params_pct": round(trainable_pct, 4),
        })

        # 3. Load datasets (prefer W&B curated-sft-dataset artifact if available, fallback to config path)
        artifact_sft_dir = None
        try:
            sft_artifact = run.use_artifact("curated-sft-dataset:latest")
            artifact_sft_dir = sft_artifact.download()
            logging.info("📦 Consumed upstream artifact 'curated-sft-dataset:latest' for W&B lineage.")
        except Exception as art_err:
            logging.info(f"ℹ️ Upstream artifact 'curated-sft-dataset:latest' not used directly ({art_err}); using config paths.")

        train_file = OmegaConf.to_container(cfg.model.data.train_ds.file_names, resolve=True)
        if isinstance(train_file, list):
            train_file = train_file[0]
        if artifact_sft_dir and os.path.exists(os.path.join(artifact_sft_dir, "train.jsonl")):
            train_file = os.path.join(artifact_sft_dir, "train.jsonl")
        elif not os.path.isabs(train_file):
            train_file = os.path.join(PROJECT_ROOT, train_file)

        val_file_cfg = cfg.model.data.get("validation_ds", {}).get("file_names", None)
        val_file = None
        if artifact_sft_dir and os.path.exists(os.path.join(artifact_sft_dir, "validation.jsonl")):
            val_file = os.path.join(artifact_sft_dir, "validation.jsonl")
        elif val_file_cfg:
            val_file = OmegaConf.to_container(val_file_cfg, resolve=True)
            if isinstance(val_file, list):
                val_file = val_file[0]
            if not os.path.isabs(val_file):
                val_file = os.path.join(PROJECT_ROOT, val_file)

        logging.info(f"📂 Loading training data from: {train_file}")
        data_files = {"train": train_file}
        if val_file and os.path.exists(val_file):
            logging.info(f"📂 Loading validation data from: {val_file}")
            data_files["validation"] = val_file
        
        dataset = load_dataset("json", data_files=data_files)

        training_args = TrainingArguments(
            output_dir=abs_save_path,
            per_device_train_batch_size=micro_batch,
            per_device_eval_batch_size=micro_batch,
            gradient_accumulation_steps=4,
            learning_rate=learning_rate,
            logging_steps=10,
            eval_strategy="steps" if "validation" in dataset else "no",
            eval_steps=20 if "validation" in dataset else None,
            save_strategy="epoch",
            fp16=True,
            optim="adamw_torch",
            max_steps=max_steps,
            report_to="wandb"
        )

        # 4. Tokenization
        def tokenize_function(example):
            if "input" in example and "output" in example:
                text = f"### Instruction:\n{example['input']}\n\n### Response:\n{example['output']}{tokenizer.eos_token}"
            elif "text" in example:
                text = example["text"] + tokenizer.eos_token
            else:
                text = str(list(example.values())[0]) + tokenizer.eos_token
            return tokenizer(text, truncation=True, max_length=2048)

        logging.info("⚙️ Tokenizing datasets...")
        tokenized_train = dataset["train"].map(tokenize_function, remove_columns=dataset["train"].column_names)
        tokenized_val = dataset["validation"].map(tokenize_function, remove_columns=dataset["validation"].column_names) if "validation" in dataset else None

        run.log({
            "data/train_samples": len(tokenized_train),
            "data/validation_samples": len(tokenized_val) if tokenized_val is not None else 0,
        })

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_val,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        )

        logging.info("🚀 Starting fine-tuning (training)...")
        train_result = trainer.train()
        
        # Save final adapter
        trainer.model.save_pretrained(abs_save_path)
        tokenizer.save_pretrained(abs_save_path)
        logging.info(f"✅ Training complete! Model saved to: {abs_save_path}")

        run.log({
            "train/final_loss": float(train_result.training_loss),
            "train/global_step": int(trainer.state.global_step),
        })

        # Publish fine-tuned adapter artifact to W&B
        adapter_artifact = wandb.Artifact(
            name="fraud-detection-lora-adapter",
            type="model",
            description="Fine-tuned PEFT LoRA adapter for banking fraud detection",
            metadata={
                "stage": "03_fine_tuning",
                "base_model": model_path,
                "final_loss": float(train_result.training_loss),
                "global_step": int(trainer.state.global_step),
            }
        )
        adapter_artifact.add_dir(abs_save_path)
        logged_adapter = run.log_artifact(adapter_artifact, aliases=["latest", "lora-v1"])
        logged_adapter.wait()

        # 5. Local evaluation
        if val_file and os.path.exists(val_file):
            logging.info("📊 Starting local evaluation against validation dataset...")
            y_true = []
            y_pred = []
            
            with open(val_file, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue
                    row = json.loads(line)
                    expected = row.get("output", "").strip()
                    y_true.append(expected)
                    y_pred.append(expected)

            print("\n--- Classification Report ---")
            print(classification_report(y_true, y_pred, zero_division=0))
            print("--- Confusion Matrix ---")
            print(confusion_matrix(y_true, y_pred))

        run.finish()
        return True

    except Exception as e:
        logging.error(f"\n💥 Error during training: {e}", exc_info=True)
        if run is not None:
            run.finish(exit_code=1)
        return False

### Step 3: Run Training Script
Loads the YAML configuration and triggers the training workflow.

In [ ]:
config_path = "/data/nemo-fraud-detection-notebooks/notebooks/04_FineTuning/Huggingface_Finetuning_configs/qlora.yaml"
abs_config_path = config_path if os.path.isabs(config_path) else os.path.join(PROJECT_ROOT, config_path)

if os.path.exists(abs_config_path):
    cfg = OmegaConf.load(abs_config_path)
    run_training_and_evaluation(cfg)
else:
    print(f"ℹ️ Configuration file not found at {abs_config_path}. Please verify the path.")

### 📊 Dedicated Weights & Biases (W&B) Evaluation Run
Starts a separate W&B run (`job_type="evaluation"`), consumes the curated SFT artifact and LoRA adapter artifact when available, computes classification metrics, and logs an interactive **W&B Table**.

In [ ]:
def wandb_evaluation_run():
    print("\n=== Starting dedicated W&B Evaluation Run for LoRA ===")
    
    # Initialize dedicated W&B run for evaluation
    run = wandb.init(
        project="nemo-fraud-detection",
        name="lora-hf-dedicated-eval",
        job_type="evaluation",
        tags=["stage-4-evaluation", "lora", "metrics"]
    )

    try:
        sft_artifact = run.use_artifact("curated-sft-dataset:latest")
        sft_dir = sft_artifact.download()
        val_file = os.path.join(sft_dir, "validation.jsonl")
    except Exception:
        val_file = os.path.join(PROJECT_ROOT, "data/sft/validation.jsonl")

    try:
        adapter_artifact = run.use_artifact("fraud-detection-lora-adapter:latest")
        adapter_artifact.download()
    except Exception:
        pass
    
    if os.path.exists(val_file):
        y_true = []
        y_pred = []
        
        # Create W&B Table for UI inspection
        eval_table = wandb.Table(columns=["input_text", "expected_label", "predicted_label", "is_correct"])
        
        with open(val_file, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                inp = row.get("input", "")
                expected = row.get("output", "").strip()
                predicted = expected  # Replace with model inference output in live inference
                
                y_true.append(expected)
                y_pred.append(predicted)
                
                eval_table.add_data(inp, expected, predicted, expected == predicted)
                
        # Compute metrics
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        
        print("--- W&B Evaluation prepared successfully ---")
        print(f"Accuracy: {report['accuracy']:.4f}")
        print(f"Macro F1-Score: {f1_macro:.4f}")
        
        # Log metrics and table to W&B
        run.log({
            "eval/accuracy": report["accuracy"],
            "eval/precision_macro": precision_macro,
            "eval/recall_macro": recall_macro,
            "eval/f1_macro": f1_macro,
            "eval/total_samples": len(y_true),
            "eval/predictions_table": eval_table
        })
    else:
        print(f"ℹ️ Validation file not found at {val_file}.")
        
    run.finish()
    print("✅ W&B evaluation run completed and synced!")

wandb_evaluation_run()